In [1]:
from pathlib import Path

import pandas as pd

In [2]:
"""
Input Schema:
===============
data/raw/split_ticket_shave.csv

year, dem_est, gop_est, margin, dem_est_votes_m, gop_est_votes_m


Output Schema:
===============
data/reference/shave.csv

year, dem_2p_norm, gop_2p_norm, margin_2p_norm, dem_est, gop_est, margin_raw, dem_est_votes_m, gop_est_votes_m

dem_2p_norm    = 100 * dem_est_votes_m / (dem_est_votes_m + gop_est_votes_m)
gop_2p_norm    = 100 * gop_est_votes_m / (dem_est_votes_m + gop_est_votes_m)
margin_2p_norm = dem_2p_norm - gop_2p_norm

NOTE: margin is renamed to margin_raw. New columns are rounded to 2 decimals to match the input.
"""

RAW_DIR = Path("../data/raw")
REFERENCE_DIR = Path("../data/reference")

shave = pd.read_csv(RAW_DIR / "split_ticket_shave.csv")
shave = shave.rename(columns={"margin": "margin_raw"})

two_party_votes = shave["dem_est_votes_m"] + shave["gop_est_votes_m"]
dem_2p_norm = 100 * shave["dem_est_votes_m"] / two_party_votes
gop_2p_norm = 100 * shave["gop_est_votes_m"] / two_party_votes

shave.insert(1, "dem_2p_norm", dem_2p_norm.round(2))
shave.insert(2, "gop_2p_norm", gop_2p_norm.round(2))
shave.insert(3, "margin_2p_norm", (dem_2p_norm - gop_2p_norm).round(2))

shave

,year,dem_2p_norm,gop_2p_norm,margin_2p_norm,dem_est,gop_est,margin_raw,dem_est_votes_m,gop_est_votes_m
0,2008,54.84,45.16,9.67,53.67,44.20,9.47,66.72,54.95
1,2010,47.39,52.61,-5.23,45.90,50.96,-5.06,40.53,45.00
2,2012,51.22,48.78,2.44,50.25,47.86,2.39,61.93,58.98
3,2014,47.42,52.58,-5.15,46.59,51.65,-5.06,37.66,41.75
4,2016,49.17,50.83,-1.67,48.01,49.64,-1.63,63.35,65.50
5,2018,53.68,46.32,7.36,53.03,45.76,7.27,61.58,53.14
6,2020,51.06,48.94,2.13,50.46,48.36,2.10,77.77,74.53
7,2022,49.20,50.80,-1.61,48.73,50.34,-1.61,53.31,55.05
8,2024,48.92,51.08,-2.17,48.35,50.50,-2.15,72.66,75.88


In [3]:
shave.to_csv(REFERENCE_DIR / "shave.csv", index=False)